# Basic FastHTML Integration Example

This notebook demonstrates the minimal setup for using cjm-tqdm-capture with FastHTML.

In [1]:
from fasthtml.common import *
from starlette.responses import StreamingResponse, JSONResponse
import uuid, time

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.LIGHT)

# Initialize monitor and runner
monitor = ProgressMonitor()
runner = JobRunner(monitor)

In [4]:
# Define a simple task that uses tqdm
def simple_task():
    from tqdm import tqdm
    import time
    
    results = []
    for i in tqdm(range(100), desc="Processing"):
        time.sleep(0.01)
        results.append(i * 2)
    return results

In [5]:
# Create minimal UI
@rt("/")
def home():
    return create_test_page(
        "Basic Progress Demo",
        Div(
            H2("Simple Progress Tracking"),
            Button("Start Task", id="start-btn", cls=combine_classes(btn, btn_colors.primary)),
            Div(
                P("Progress:", cls=combine_classes(font_weight.bold, m.t(4))),
                Progress(id="progress", max="100", value="0", cls=combine_classes(progress, progress_colors.primary, w.full)),
                P("Ready", id="status", cls=combine_classes(m.t(2), font_size.sm)),
                cls=str(m.t(4))
            ),
            cls=str(p(8))
        ),
        Script("""
            const btn = document.getElementById('start-btn');
            const progress = document.getElementById('progress');
            const status = document.getElementById('status');
            
            btn.onclick = async () => {
                btn.disabled = true;
                status.textContent = 'Starting...';
                
                // Create job
                const resp = await fetch('/api/jobs', {method: 'POST'});
                const data = await resp.json();
                
                // Connect SSE
                const sse = new EventSource(`/api/jobs/${data.job_id}/stream`);
                
                sse.onmessage = (e) => {
                    const payload = JSON.parse(e.data);
                    progress.value = payload.progress || 0;
                    
                    if (payload.bars) {
                        const bar = Object.values(payload.bars)[0];
                        if (bar) {
                            status.textContent = `${bar.desc}: ${bar.pct.toFixed(1)}% (${bar.cur}/${bar.tot})`;
                        }
                    }
                    
                    if (payload.completed) {
                        status.textContent = 'Completed!';
                        btn.disabled = false;
                        sse.close();
                    }
                };
                
                sse.onerror = () => {
                    status.textContent = 'Error or completed';
                    btn.disabled = false;
                    sse.close();
                };
            };
        """)
    )

In [6]:
# API endpoints
@rt("/api/jobs", methods=["POST"])
def create_job():
    job_id = str(uuid.uuid4())
    runner.start(
        job_id, 
        simple_task,
        patch_kwargs={
            "min_delta_pct": 5,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    return JSONResponse({"job_id": job_id})

@rt("/api/jobs/{job_id}/stream")
def stream_progress(job_id: str):
    gen = sse_stream(
        monitor, 
        job_id,
        interval=0.1,
        heartbeat=10.0
    )
    return StreamingResponse(gen, media_type="text/event-stream")

In [7]:
# Start server
server = start_test_server(app)
display(HTMX())

In [8]:
# Stop server when done
server.stop()